# Jira Assistant: description → ticket, design and probable implementation

Given a short, informal description of a feature or bug, this assistant:

1. Retrieves similar **past Jira tickets** (including how they were actually implemented) as grounding context — the RAG step
2. Drafts a well-formed **Jira ticket** (summary, type, priority, components, labels, description, acceptance criteria, story point estimate)
3. Proposes a **technical design**
4. Sketches a **probable implementation** plan

This follows the same simple, keyword-based retrieval approach as `introduction_rag.ipynb`, applied to a CSV export of historical Jira tickets instead of employee bios.

In [1]:
# os / pandas: read the .env file and the Jira CSV export
import os
import pandas as pd
from dotenv import load_dotenv
# gradio: quick web chat UI for the demo at the end of the notebook
import gradio as gr
# the official OpenAI client, used to call the chat completions API
from openai import OpenAI

/Users/robertkigobe/Documents/My_Research/jira_rag_assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setting up

In [2]:
# override=True: re-read .env even if these vars are already set in the shell
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    # only print the prefix — never print the full key
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# small, cheap model — good enough for drafting tickets; swap for gpt-4.1 for higher-quality designs
MODEL = "gpt-4.1-nano"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


### Let's read the Jira export into a knowledge base

Each past ticket becomes one document: its summary, type, components, labels, description and resolution comments (which often describe *how* it was actually built).

In [3]:
# fillna("") avoids NaN showing up as "nan" once we format rows into text below
df = pd.read_csv("jira_fictitious_data.csv").fillna("")
df.shape

(50, 22)

In [4]:
# Flatten one CSV row into a single text blob the LLM can read as "one document".
# Comment (resolution notes) is included because it often says HOW the ticket was actually built —
# that's the part we want the model to reuse later.
def format_ticket(row):
    return (
        f"Issue key: {row['Issue key']}\n"
        f"Type: {row['Issue Type']} | Priority: {row['Priority']} | Story Points: {row['Story Points']}\n"
        f"Component/s: {row['Component/s']} | Labels: {row['Labels']}\n"
        f"Summary: {row['Summary']}\n"
        f"Description: {row['Description']}\n"
        f"Resolution notes: {row['Comment']}"
    )

# our "knowledge base": Issue key -> formatted ticket text, same shape as the
# name -> bio dict in introduction_rag.ipynb
knowledge = {row["Issue key"]: format_ticket(row) for _, row in df.iterrows()}
len(knowledge)

50

In [5]:
# sanity check: inspect one formatted ticket document
knowledge["PLAT-4521"]

'Issue key: PLAT-4521\nType: Story | Priority: High | Story Points: 5.0\nComponent/s: API Gateway | Labels: backend;security\nSummary: Implement OAuth2 token refresh logic for API gateway\nDescription: The current API gateway does not handle token refresh gracefully. When an OAuth2 access token expires mid-request, the gateway returns a 401 to the client instead of transparently refreshing the token using the refresh token stored in the session. This causes intermittent failures for long-running batch operations. Acceptance criteria: 1) Gateway intercepts 401 responses from downstream services 2) Automatically attempts token refresh using stored refresh token 3) Retries the original request with the new token 4) If refresh fails, returns 401 to client with appropriate error message 5) Logs refresh events for monitoring\nResolution notes: 22/Jan/26 2:00 PM;jmartinez;Implemented the refresh interceptor in the gateway middleware. Added unit tests covering the happy path and failure scenar

### Retrieval

`introduction_rag.ipynb` matched single words straight against dictionary keys (people's surnames, product names). Jira tickets don't have one-word keys, so instead we score each ticket by how many words it shares with the incoming description, and keep the strongest matches — still just plain keyword overlap, no embeddings involved.

In [6]:
# strip punctuation, lowercase, split into a set of words — deliberately crude,
# same "good enough" philosophy as introduction_rag.ipynb's word matching
def tokenize(text):
    cleaned = ''.join(ch for ch in text if ch.isalnum() or ch.isspace())
    return set(cleaned.lower().split())

# pre-compute each ticket's word set once, so scoring against it later is cheap
ticket_words = {key: tokenize(text) for key, text in knowledge.items()}

In [7]:
# score = size of the word-set intersection between the incoming description and each ticket;
# higher overlap = more relevant. This is the actual "retrieval" step of the RAG pipeline.
def get_relevant_tickets(message, top_k=3):
    message_words = tokenize(message)
    scored = [(key, len(words & message_words)) for key, words in ticket_words.items()]
    scored = [(key, score) for key, score in scored if score > 0]  # drop zero-overlap tickets
    scored.sort(key=lambda pair: pair[1], reverse=True)  # best matches first
    return [knowledge[key] for key, _ in scored[:top_k]]  # only send the top_k as context

In [8]:
# try it standalone, before wiring it into a prompt — should surface the dark-mode / design-system tickets
get_relevant_tickets("Add dark mode toggle to the notifications settings page")

['Issue key: UXD-2103\nType: Epic | Priority: High | Story Points: 13.0\nComponent/s: Design System | Labels: frontend;design-system;accessibility\nSummary: Design system: Add dark mode variants to component library\nDescription: Executive request: Implement comprehensive dark mode support across the entire component library. This is a customer-facing requirement as 67% of surveyed enterprise users prefer dark mode during extended sessions. Scope: All 48 components in the current design system need dark mode variants. Each component must: 1) Support automatic system preference detection 2) Allow manual toggle override 3) Maintain WCAG 2.1 AA contrast ratios in both modes 4) Animate transitions smoothly (200ms ease) 5) Persist user preference across sessions. Dependencies: Needs updated design tokens from the brand team. Estimated effort: 6-8 sprints for full coverage.\nResolution notes: ',
 'Issue key: UXD-2104\nType: Sub-task | Priority: High | Story Points: 3.0\nComponent/s: Design S

In [9]:
# turn the retrieved tickets into the block of text that gets appended to the system prompt
def additional_context(message):
    relevant_tickets = get_relevant_tickets(message)
    if not relevant_tickets:
        return "There are no similar past tickets relevant to this request."
    result = "The following past tickets may be relevant precedent (reuse their patterns, estimates and pitfalls where appropriate):\n\n"
    result += "\n\n---\n\n".join(relevant_tickets)
    return result

In [10]:
# preview exactly what context the LLM will see for this description
print(additional_context("Add dark mode toggle to the notifications settings page"))

The following past tickets may be relevant precedent (reuse their patterns, estimates and pitfalls where appropriate):

Issue key: UXD-2103
Type: Epic | Priority: High | Story Points: 13.0
Component/s: Design System | Labels: frontend;design-system;accessibility
Summary: Design system: Add dark mode variants to component library
Description: Executive request: Implement comprehensive dark mode support across the entire component library. This is a customer-facing requirement as 67% of surveyed enterprise users prefer dark mode during extended sessions. Scope: All 48 components in the current design system need dark mode variants. Each component must: 1) Support automatic system preference detection 2) Allow manual toggle override 3) Maintain WCAG 2.1 AA contrast ratios in both modes 4) Animate transitions smoothly (200ms ease) 5) Persist user preference across sessions. Dependencies: Needs updated design tokens from the brand team. Estimated effort: 6-8 sprints for full coverage.
Resol

### The system prompt

Instructs the model to turn a rough description into a ticket, a design, and a probable implementation — grounded in the retrieved precedent above.

In [11]:
# The retrieved ticket context (from additional_context) gets concatenated onto the end of
# this string before it's sent as the system message — see chat() below.
SYSTEM_PREFIX = """
You are a senior engineering lead triaging incoming work for a software team.
You are given a short, informal description of a feature request or bug, plus context on similar
past Jira tickets from this team (including how they were actually resolved).

Using that precedent where relevant, produce a response with exactly three sections:

## 🎫 Jira Ticket
- **Summary**: one line
- **Issue Type**: Story / Bug / Task / Sub-task / Epic
- **Priority**: Low / Medium / High / Critical
- **Component/s** and **Labels**: inferred from the description and precedent
- **Description**: a clear problem statement
- **Acceptance Criteria**: a numbered list
- **Story Point estimate**: with a one-line rationale, informed by similar past tickets' points if available

## 🏗️ Technical Design
Approach, affected components/services, key decisions and trade-offs, risks or open questions.

## 🔧 Probable Implementation
A concrete, numbered implementation plan (files/modules likely touched, key steps, tests to add).
Where a similar past ticket exists, reuse its approach explicitly instead of inventing a new one.

If the description is vague, state your assumptions rather than asking clarifying questions.

Relevant context:
"""

### Putting it together

In [12]:
# The full pipeline in one function: retrieve -> build system prompt -> call the LLM.
# `history` follows Gradio's messages format ([{"role": ..., "content": ...}, ...]) so this
# plugs directly into gr.ChatInterface below, and also lets a user ask follow-up refinements.
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [13]:
# end-to-end test outside the UI: empty history = a fresh, one-shot request
print(chat("Users report the export-to-CSV button on the reporting dashboard times out for large date ranges", []))

## 🎫 Jira Ticket
- **Summary**: Improve export-to-CSV performance for large date ranges on reporting dashboard
- **Issue Type**: Bug
- **Priority**: High
- **Component/s**: Dashboard | Reporting | Frontend;Backend
- **Description**: Users experience timeouts when exporting large datasets via the CSV export button on the reporting dashboard, especially with extensive date ranges. The current implementation loads all data into memory before writing, leading to slow response times and timeouts during large exports.
- **Acceptance Criteria**:
  1. Export operation completes within acceptable time (e.g., ≤30 seconds) for typical large date ranges.
  2. For very large data, the system gracefully handles the export or provides a download link via background processing.
  3. The CSV export feature respects pagination during data retrieval to prevent memory overload.
  4. Users receive clear feedback if export cannot be completed immediately (e.g., progress indicator or background job notificat

### Now bring this up in Gradio using the Chat interface

Paste in a rough Jira description; each reply is a full ticket + design + probable implementation.

In [14]:
# launches a local web UI in a new browser tab; blocks the notebook kernel while running.
# Interrupt the cell (■ button) to stop the server and get control of the notebook back.
# note: gradio 6.x's ChatInterface no longer takes `type=` — "messages" format is now the only option.
view = gr.ChatInterface(
    chat,
    title="Jira Assistant",
    description="Describe a feature or bug and get a drafted ticket, technical design, and probable implementation.",
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
